## Funciones

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from spectral import open_image


def extract_specimen_only_from_hsi_path(
    hdr_path,
    r_nm=650,
    g_nm=550,
    b_nm=450,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=False
):
    """
    Carga una imagen hiperespectral ENVI desde una ruta .hdr y devuelve una pseudo-RGB
    donde solo se ve el espécimen; el resto queda negro.

    Parámetros
    ----------
    hdr_path : str
        Ruta al archivo .hdr de la imagen hiperespectral.

    r_nm, g_nm, b_nm : float
        Longitudes de onda usadas para construir la pseudo-RGB.

    grow_pixels : int
        Número de píxeles para agrandar ligeramente el contorno del espécimen.
        Útil para evitar cortar tejido.

    show_original : bool
        Si True, plotea la pseudo-RGB original.

    show_result : bool
        Si True, plotea la imagen con solo el espécimen.

    return_mask : bool
        Si True, devuelve también la máscara binaria del espécimen.

    Returns
    -------
    specimen_only_rgb : np.ndarray
        Imagen RGB uint8 con solo el espécimen visible y el resto negro.

    Opcionalmente:
    mask_specimen : np.ndarray
        Máscara uint8 del espécimen, con valores 0 y 255.
    """

    # ============================================================
    # 1. Cargar cubo HSI
    # ============================================================
    img = open_image(hdr_path)
    cube = np.asarray(img.load(), dtype=np.float32)

    wavelengths = np.array(
        [float(w) for w in img.metadata["wavelength"]],
        dtype=np.float32
    )

    # ============================================================
    # 2. Funciones internas
    # ============================================================
    def find_nearest_band(wavelengths, target_nm):
        wavelengths_arr = np.asarray(wavelengths, dtype=float)
        idx = np.argmin(np.abs(wavelengths_arr - target_nm))
        return idx

    def robust_normalize(channel, p_low=2, p_high=98):
        ch = channel.astype(np.float32)

        lo = np.percentile(ch, p_low)
        hi = np.percentile(ch, p_high)

        if hi <= lo:
            return np.zeros_like(ch, dtype=np.float32)

        ch = (ch - lo) / (hi - lo)
        ch = np.clip(ch, 0, 1)

        return ch

    # ============================================================
    # 3. Crear pseudo-RGB
    # ============================================================
    r_idx = find_nearest_band(wavelengths, r_nm)
    g_idx = find_nearest_band(wavelengths, g_nm)
    b_idx = find_nearest_band(wavelengths, b_nm)

    r = robust_normalize(cube[:, :, r_idx])
    g = robust_normalize(cube[:, :, g_idx])
    b = robust_normalize(cube[:, :, b_idx])

    pseudo_rgb = np.stack([r, g, b], axis=-1)
    rgb_u8 = (np.clip(pseudo_rgb, 0, 1) * 255).astype(np.uint8)

    # ============================================================
    # 4. Detectar caja azul
    # ============================================================
    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

    R = rgb_u8[:, :, 0].astype(np.int16)
    G = rgb_u8[:, :, 1].astype(np.int16)
    B = rgb_u8[:, :, 2].astype(np.int16)

    V = hsv[:, :, 2]

    lower_blue = np.array([85, 10, 40], dtype=np.uint8)
    upper_blue = np.array([125, 180, 255], dtype=np.uint8)

    mask_hsv = cv2.inRange(hsv, lower_blue, upper_blue)

    mask_dominance = (
        ((B - R) > 8) &
        ((G - R) > 3) &
        (V > 50)
    )

    mask_blue = mask_hsv & (mask_dominance.astype(np.uint8) * 255)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 35))
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))

    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel_open)
    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel_close)
    mask_blue = cv2.dilate(mask_blue, kernel_dilate, iterations=1)

    if np.count_nonzero(mask_blue) == 0:
        raise ValueError("No se detectó la caja azul. Revisa los umbrales HSV.")

    # ============================================================
    # 5. Quedarse con el componente azul más grande
    # ============================================================
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_blue,
        connectivity=8
    )

    if num_labels <= 1:
        raise ValueError("No se encontró ningún componente azul válido.")

    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_label = 1 + np.argmax(areas)

    mask_box_blue = (labels == largest_label).astype(np.uint8) * 255

    # ============================================================
    # 6. Contorno interno de la caja azul = espécimen
    # ============================================================
    contours, hierarchy = cv2.findContours(
        mask_box_blue,
        cv2.RETR_CCOMP,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if hierarchy is None:
        raise ValueError("No se encontraron contornos en la máscara de la caja azul.")

    hierarchy = hierarchy[0]

    inner_contours = [
        cnt for cnt, h in zip(contours, hierarchy)
        if h[3] != -1
    ]

    if len(inner_contours) == 0:
        raise ValueError("No se encontró ningún hueco interno en la caja azul.")

    specimen_contour = max(inner_contours, key=cv2.contourArea)

    mask_specimen = np.zeros_like(mask_box_blue, dtype=np.uint8)
    cv2.drawContours(
        mask_specimen,
        [specimen_contour],
        -1,
        255,
        thickness=cv2.FILLED
    )

    # Suavizar pequeñas irregularidades
    kernel_smooth = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_specimen = cv2.morphologyEx(
        mask_specimen,
        cv2.MORPH_CLOSE,
        kernel_smooth
    )

    # Agrandar ligeramente el contorno
    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * grow_pixels + 1, 2 * grow_pixels + 1)
        )

        mask_specimen = cv2.dilate(
            mask_specimen,
            kernel_grow,
            iterations=1
        )

    # ============================================================
    # 7. Aplicar máscara: espécimen visible, resto negro
    # ============================================================
    specimen_only_rgb = rgb_u8.copy()
    specimen_only_rgb[mask_specimen == 0] = 0

    # ============================================================
    # 8. Plots
    # ============================================================
    if show_original and show_result:
        plt.figure(figsize=(12, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(specimen_only_rgb)
        plt.title("Solo espécimen")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis("off")
        plt.show()

    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(specimen_only_rgb)
        plt.title("Solo espécimen")
        plt.axis("off")
        plt.show()

    if return_mask:
        return specimen_only_rgb, mask_specimen

    return specimen_only_rgb

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import openslide


def extract_he_tissue_contour_image(
    slide_path,
    max_dim=1800,
    sat_thresh=8,
    val_thresh=253,
    od_thresh=0.025,
    min_area=500,
    open_kernel_size=3,
    close_kernel_size=45,
    grow_pixels=8,
    use_convex_hull=False,
    show_original=True,
    show_result=True,
    debug=False,
    return_mask=False,
    return_info=False
):
    """
    Extrae el tejido de una H&E tipo .mrxs usando el contorno externo del tejido.

    Por defecto devuelve solo:
        tissue_only_rgb

    Opcionalmente puede devolver:
        mask_tissue
        info

    Parameters
    ----------
    slide_path : str or Path
        Ruta al archivo .mrxs.

    max_dim : int
        Tamaño máximo del lado largo usado para leer la imagen a baja resolución.

    sat_thresh : int
        Umbral de saturación HSV. Más bajo detecta tejido más pálido.

    val_thresh : int
        Umbral de brillo HSV. Más alto permite zonas más claras.

    od_thresh : float
        Umbral de optical density. Más bajo detecta tejido muy claro.

    min_area : int
        Área mínima de componentes a conservar.

    open_kernel_size : int
        Kernel para eliminar ruido pequeño.

    close_kernel_size : int
        Kernel para cerrar huecos y unir zonas del tejido.

    grow_pixels : int
        Dilatación final del contorno para no cortar tejido.

    use_convex_hull : bool
        Si True, usa convex hull. Es más conservador, puede meter más fondo.

    show_original : bool
        Plotea la H&E original leída.

    show_result : bool
        Plotea el resultado final.

    debug : bool
        Muestra pasos intermedios.

    return_mask : bool
        Si True, devuelve también la máscara final.

    return_info : bool
        Si True, devuelve también información auxiliar.
    """

    slide_path = Path(slide_path)

    if not slide_path.exists():
        raise FileNotFoundError(f"No existe el archivo: {slide_path}")

    # ============================================================
    # 1. Cargar slide a baja resolución
    # ============================================================
    slide = openslide.OpenSlide(str(slide_path))
    level_dims = slide.level_dimensions

    chosen_level = None
    for i, (w, h) in enumerate(level_dims):
        if max(w, h) <= max_dim:
            chosen_level = i
            break

    if chosen_level is None:
        chosen_level = len(level_dims) - 1

    level_w, level_h = level_dims[chosen_level]

    rgb_pil = slide.read_region(
        location=(0, 0),
        level=chosen_level,
        size=(level_w, level_h)
    ).convert("RGB")

    slide.close()

    rgb_u8 = np.array(rgb_pil, dtype=np.uint8)

    # ============================================================
    # 2. Mapas HSV y optical density
    # ============================================================
    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

    S = hsv[:, :, 1]
    V = hsv[:, :, 2]

    rgb_float = rgb_u8.astype(np.float32)
    od = -np.log((rgb_float + 1.0) / 255.0)
    od_sum = od.sum(axis=2)

    # ============================================================
    # 3. Máscara inicial: tejido teñido o tejido pálido
    # ============================================================
    mask_sat = (
        (S > sat_thresh) &
        (V < val_thresh) &
        (V > 20)
    ).astype(np.uint8) * 255

    mask_od = (
        (od_sum > od_thresh) &
        (V < val_thresh) &
        (V > 20)
    ).astype(np.uint8) * 255

    mask_raw = cv2.bitwise_or(mask_sat, mask_od)

    # ============================================================
    # 4. Morfología
    # ============================================================
    mask_morph = mask_raw.copy()

    if open_kernel_size > 0:
        kernel_open = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (open_kernel_size, open_kernel_size)
        )
        mask_morph = cv2.morphologyEx(mask_morph, cv2.MORPH_OPEN, kernel_open)

    if close_kernel_size > 0:
        kernel_close = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (close_kernel_size, close_kernel_size)
        )
        mask_morph = cv2.morphologyEx(mask_morph, cv2.MORPH_CLOSE, kernel_close)

    # ============================================================
    # 5. Componentes conectados
    # ============================================================
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_morph,
        connectivity=8
    )

    mask_components = np.zeros_like(mask_morph, dtype=np.uint8)

    for label_id in range(1, num_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area >= min_area:
            mask_components[labels == label_id] = 255

    if np.count_nonzero(mask_components) == 0:
        raise ValueError(
            "No se detectó tejido. Prueba sat_thresh=5, od_thresh=0.012, val_thresh=255."
        )

    # ============================================================
    # 6. Extraer contorno externo y rellenarlo
    # ============================================================
    contours, _ = cv2.findContours(
        mask_components,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        raise ValueError("No se encontraron contornos de tejido.")

    # Normalmente el tejido principal es el mayor contorno
    largest_contour = max(contours, key=cv2.contourArea)

    mask_contour = np.zeros_like(mask_components, dtype=np.uint8)

    if use_convex_hull:
        hull = cv2.convexHull(largest_contour)
        cv2.drawContours(mask_contour, [hull], -1, 255, thickness=cv2.FILLED)
    else:
        cv2.drawContours(mask_contour, [largest_contour], -1, 255, thickness=cv2.FILLED)

    # Dilatar ligeramente para no cortar borde
    mask_final = mask_contour.copy()

    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * grow_pixels + 1, 2 * grow_pixels + 1)
        )
        mask_final = cv2.dilate(mask_final, kernel_grow, iterations=1)

    # ============================================================
    # 7. Aplicar máscara: tejido visible, resto negro
    # ============================================================
    tissue_only_rgb = rgb_u8.copy()
    tissue_only_rgb[mask_final == 0] = 0

    # ============================================================
    # 8. Plots
    # ============================================================
    if debug:
        od_vis = (od_sum - od_sum.min()) / (od_sum.max() - od_sum.min() + 1e-8)

        plt.figure(figsize=(18, 10))

        plt.subplot(2, 4, 1)
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")

        plt.subplot(2, 4, 2)
        plt.imshow(S, cmap="gray")
        plt.title("Saturación HSV")
        plt.axis("off")

        plt.subplot(2, 4, 3)
        plt.imshow(od_vis, cmap="gray")
        plt.title("Optical density")
        plt.axis("off")

        plt.subplot(2, 4, 4)
        plt.imshow(mask_raw, cmap="gray")
        plt.title("Máscara inicial")
        plt.axis("off")

        plt.subplot(2, 4, 5)
        plt.imshow(mask_morph, cmap="gray")
        plt.title("Después morfología")
        plt.axis("off")

        plt.subplot(2, 4, 6)
        plt.imshow(mask_components, cmap="gray")
        plt.title("Componentes filtrados")
        plt.axis("off")

        plt.subplot(2, 4, 7)
        plt.imshow(mask_final, cmap="gray")
        plt.title("Máscara final por contorno")
        plt.axis("off")

        plt.subplot(2, 4, 8)
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original and show_result:
        plt.figure(figsize=(10, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")
        plt.show()

    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")
        plt.show()

    # ============================================================
    # 9. Return limpio
    # ============================================================
    outputs = [tissue_only_rgb]

    if return_mask:
        outputs.append(mask_final)

    if return_info:
        info = {
            "chosen_level": chosen_level,
            "read_dimensions": (level_w, level_h),
            "sat_thresh": sat_thresh,
            "val_thresh": val_thresh,
            "od_thresh": od_thresh,
            "grow_pixels": grow_pixels,
            "use_convex_hull": use_convex_hull,
            "mask_area_px": int(np.count_nonzero(mask_final)),
        }
        outputs.append(info)

    if len(outputs) == 1:
        return tissue_only_rgb

    return tuple(outputs)

## Seguimos

Primeros pasos

Imágenes q usaremos

In [ ]:
# Datos\SB013\SB013\SB013_ink_annotation.png
# Datos\SB013\SB013\H&E_python_EDU_creado\SB013.mrxs   
# Datos\SB013\SB013\H&E_python_EDU_creado\H&E.xml
# Datos\SB013\SB013\SB013_001\SB013_nrm.hdr

In [ ]:
from pathlib import Path
import re
import unicodedata
import difflib
import xml.etree.ElementTree as ET

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import openslide
from matplotlib.patches import Polygon, Rectangle


def extract_he_hsi_ink_annotations(
    he_xml_path,
    hsi_ink_png_path,
    keep_inks=("green", "red", "blue"),
    show_debug=True,
):
    """
    Extrae las anotaciones de tintas en:

    1) H&E.xml:
       - Lee polígonos anotados por el patólogo.
       - Se queda solo con anotaciones de tinta: green/red/blue.
       - Soporta errores tipo 'Greeen'.

    2) SB013_ink_annotation.png:
       - Detecta rectángulos/cuadrados dibujados a mano en verde/rojo/azul.
       - Devuelve bbox, centroide y puntos del rectángulo.

    Devuelve:
        result = {
            "HE":  lista de anotaciones H&E,
            "HSI": lista de anotaciones HSI,
            "common_inks": tintas presentes en ambas modalidades,
            "meta": información de tamaños/coordenadas
        }
    """

    # ============================================================
    # Helpers de texto
    # ============================================================

    ink_aliases = {
        "green": ["green", "greeen", "verde"],
        "red": ["red", "rojo", "roja"],
        "blue": ["blue", "azul"],
    }

    def normalize_text(s):
        s = unicodedata.normalize("NFKD", str(s))
        s = s.encode("ascii", "ignore").decode("ascii")
        s = s.lower().strip()
        s = re.sub(r"[^a-z0-9]+", " ", s)
        return s.strip()

    def canonical_ink_name(name):
        """
        Convierte nombres tipo 'Greeen', 'RED', 'verde', etc.
        a: green/red/blue.
        """
        n = normalize_text(name)
        tokens = n.split()

        for canonical, aliases in ink_aliases.items():
            for alias in aliases:
                alias_n = normalize_text(alias)

                if alias_n == n or alias_n in tokens:
                    return canonical

                # Tolerancia a errores tipo Greeen
                if len(tokens) == 1:
                    ratio = difflib.SequenceMatcher(None, tokens[0], alias_n).ratio()
                    if ratio > 0.78:
                        return canonical

        return None

    # ============================================================
    # Helpers geométricos
    # ============================================================

    def polygon_features(points):
        pts = np.asarray(points, dtype=np.float64)

        x = pts[:, 0]
        y = pts[:, 1]

        bbox = (
            float(x.min()),
            float(y.min()),
            float(x.max()),
            float(y.max()),
        )

        centroid = pts.mean(axis=0)

        if len(pts) >= 3:
            area = abs(cv2.contourArea(pts.astype(np.float32)))
        else:
            area = 0.0

        return {
            "points": pts,
            "centroid": centroid,
            "bbox": bbox,
            "area": area,
        }

    # ============================================================
    # 1. Leer anotaciones del H&E.xml
    # ============================================================

    def read_he_xml_annotations(xml_path):
        xml_path = Path(xml_path)
        root = ET.parse(xml_path).getroot()

        # Tamaños declarados en el XML
        source_roi = root.find(".//source/region_of_interest")
        destination_roi = root.find(".//destination/region_of_interest")

        source_size = None
        destination_size = None

        if source_roi is not None:
            source_size = (
                int(float(source_roi.attrib["width"])),
                int(float(source_roi.attrib["height"])),
            )

        if destination_roi is not None:
            destination_size = (
                int(float(destination_roi.attrib["width"])),
                int(float(destination_roi.attrib["height"])),
            )

        he_annotations = []

        for ann in root.findall(".//annotation"):
            raw_name = ann.attrib.get("name", "")
            ink = canonical_ink_name(raw_name)

            if ink is None:
                continue

            if ink not in keep_inks:
                continue

            points = []
            for p in ann.findall(".//p"):
                x = float(p.attrib["x"])
                y = float(p.attrib["y"])
                points.append((x, y))

            if len(points) == 0:
                continue

            features = polygon_features(points)

            item = {
                "modality": "HE",
                "ink": ink,
                "raw_name": raw_name,
                "type": ann.attrib.get("type", ""),
                "color_bgr_xml": ann.attrib.get("color_bgr", ""),
                **features,
            }

            he_annotations.append(item)

        # ------------------------------------------------------------
        # OJO:
        # En tu XML, las coordenadas de las anotaciones parecen encajar
        # con source_size, no con destination_size, porque hay valores y
        # mayores que destination height.
        # Lo inferimos automáticamente.
        # ------------------------------------------------------------

        coord_size = source_size
        coord_system = "source"

        if len(he_annotations) > 0 and destination_size is not None:
            all_points = np.concatenate(
                [a["points"] for a in he_annotations],
                axis=0,
            )

            max_x = all_points[:, 0].max()
            max_y = all_points[:, 1].max()

            if max_x <= destination_size[0] and max_y <= destination_size[1]:
                coord_size = destination_size
                coord_system = "destination"

        # Añadir coordenadas normalizadas
        if coord_size is not None:
            W, H = coord_size

            for a in he_annotations:
                cx, cy = a["centroid"]
                x1, y1, x2, y2 = a["bbox"]

                a["centroid_norm"] = np.array([cx / W, cy / H], dtype=float)
                a["bbox_norm"] = (
                    x1 / W,
                    y1 / H,
                    x2 / W,
                    y2 / H,
                )

        he_meta = {
            "xml_path": str(xml_path),
            "source_size": source_size,
            "destination_size": destination_size,
            "coord_system_inferred": coord_system,
            "coord_size_used": coord_size,
        }

        return he_annotations, he_meta

    # ============================================================
    # 2. Detectar rectángulos de tinta en PNG de HSI
    # ============================================================

    def mask_ink_color_rgb(img_rgb, ink):
        """
        Segmentación simple por dominancia de color.

        Está pensada para anotaciones artificiales tipo Photoshop,
        no para segmentar tejido real.
        """
        r = img_rgb[:, :, 0].astype(np.int16)
        g = img_rgb[:, :, 1].astype(np.int16)
        b = img_rgb[:, :, 2].astype(np.int16)

        if ink == "green":
            # En tu PNG el verde es aprox RGB=(0,180,75)
            mask = (g >= 120) & (g - r >= 70) & (g - b >= 35)

        elif ink == "red":
            mask = (r >= 150) & (r - g >= 70) & (r - b >= 70)

        elif ink == "blue":
            mask = (b >= 150) & (b - r >= 60) & (b - g >= 40)

        else:
            raise ValueError(f"Tinta no soportada: {ink}")

        return mask.astype(np.uint8) * 255

    def component_rectangle_candidates(
        mask,
        min_area=80,
        min_w=10,
        min_h=10,
        max_fill_frac=0.45,
    ):
        """
        Busca componentes conectados que parezcan bordes de rectángulos.

        max_fill_frac bajo porque un rectángulo dibujado como borde
        no está relleno.
        """
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        mask_clean = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

        n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
            mask_clean,
            connectivity=8,
        )

        candidates = []

        for i in range(1, n_labels):
            x, y, w, h, area = stats[i]

            if area < min_area:
                continue

            if w < min_w or h < min_h:
                continue

            bbox_area = w * h
            fill_frac = area / bbox_area

            # Un borde rectangular ocupa poca fracción de su bbox.
            # Una región rellena o tejido ocupa más.
            if fill_frac > max_fill_frac:
                continue

            candidate = {
                "bbox": (
                    int(x),
                    int(y),
                    int(x + w - 1),
                    int(y + h - 1),
                ),
                "centroid": np.array(centroids[i], dtype=float),
                "area": int(area),
                "fill_frac": float(fill_frac),
                "component_id": int(i),
            }

            candidates.append(candidate)

        return candidates, mask_clean

    def detect_hsi_png_annotations(png_path):
        png_path = Path(png_path)

        img_bgr = cv2.imread(str(png_path), cv2.IMREAD_COLOR)

        if img_bgr is None:
            raise FileNotFoundError(f"No se pudo abrir la imagen: {png_path}")

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        H, W = img_rgb.shape[:2]

        hsi_annotations = []
        debug_masks = {}

        for ink in keep_inks:
            mask = mask_ink_color_rgb(img_rgb, ink)
            candidates, mask_clean = component_rectangle_candidates(mask)

            debug_masks[ink] = mask_clean

            for c in candidates:
                x1, y1, x2, y2 = c["bbox"]

                rect_points = np.array(
                    [
                        [x1, y1],
                        [x2, y1],
                        [x2, y2],
                        [x1, y2],
                        [x1, y1],
                    ],
                    dtype=float,
                )

                cx, cy = c["centroid"]

                item = {
                    "modality": "HSI",
                    "ink": ink,
                    "raw_name": ink,
                    "type": "detected_rectangle",
                    "points": rect_points,
                    "centroid": c["centroid"],
                    "bbox": c["bbox"],
                    "centroid_norm": np.array([cx / W, cy / H], dtype=float),
                    "bbox_norm": (
                        x1 / W,
                        y1 / H,
                        x2 / W,
                        y2 / H,
                    ),
                    "area": c["area"],
                    "fill_frac": c["fill_frac"],
                }

                hsi_annotations.append(item)

        hsi_meta = {
            "png_path": str(png_path),
            "image_size": (W, H),
            "image_rgb": img_rgb,
            "debug_masks": debug_masks,
        }

        return hsi_annotations, hsi_meta

    # ============================================================
    # Ejecutar extracción
    # ============================================================

    he_annotations, he_meta = read_he_xml_annotations(he_xml_path)
    hsi_annotations, hsi_meta = detect_hsi_png_annotations(hsi_ink_png_path)

    he_inks = set(a["ink"] for a in he_annotations)
    hsi_inks = set(a["ink"] for a in hsi_annotations)

    common_inks = sorted(he_inks & hsi_inks)

    result = {
        "HE": he_annotations,
        "HSI": hsi_annotations,
        "common_inks": common_inks,
        "meta": {
            "HE": he_meta,
            "HSI": {
                "png_path": hsi_meta["png_path"],
                "image_size": hsi_meta["image_size"],
            },
        },
    }

    # ============================================================
    # Debug plots
    # ============================================================

    if show_debug:
        print("\n==============================")
        print("H&E XML")
        print("==============================")
        print("source_size:", he_meta["source_size"])
        print("destination_size:", he_meta["destination_size"])
        print("coord_system_inferred:", he_meta["coord_system_inferred"])
        print("coord_size_used:", he_meta["coord_size_used"])

        print("\nAnotaciones H&E encontradas:")
        for a in he_annotations:
            print(
                f"  - ink={a['ink']:>5s} | raw_name={a['raw_name']} | "
                f"centroid={a['centroid']} | bbox={a['bbox']}"
            )

        print("\n==============================")
        print("HSI PNG")
        print("==============================")
        print("image_size:", hsi_meta["image_size"])

        print("\nRectángulos HSI encontrados:")
        for a in hsi_annotations:
            print(
                f"  - ink={a['ink']:>5s} | centroid={a['centroid']} | "
                f"bbox={a['bbox']} | area={a['area']} | fill={a['fill_frac']:.3f}"
            )

        print("\nTintas comunes HE-HSI:", common_inks)

        # Plot H&E polygons en sistema de coordenadas XML
        if len(he_annotations) > 0:
            plt.figure(figsize=(6, 8))
            for a in he_annotations:
                pts = a["points"]
                plt.plot(pts[:, 0], pts[:, 1], marker="o", label=f"HE {a['ink']}")

                cx, cy = a["centroid"]
                plt.scatter([cx], [cy], s=80)

            plt.gca().invert_yaxis()
            plt.axis("equal")
            plt.title("Anotaciones de tinta en H&E.xml")
            plt.legend()
            plt.show()

        # Plot HSI con rectángulos detectados
        img_rgb = hsi_meta["image_rgb"].copy()

        plt.figure(figsize=(10, 8))
        plt.imshow(img_rgb)

        for a in hsi_annotations:
            x1, y1, x2, y2 = a["bbox"]

            if a["ink"] == "green":
                color = "lime"
            elif a["ink"] == "red":
                color = "red"
            elif a["ink"] == "blue":
                color = "blue"
            else:
                color = "yellow"

            plt.gca().add_patch(
                plt.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    edgecolor=color,
                    linewidth=3,
                )
            )

            cx, cy = a["centroid"]
            plt.scatter([cx], [cy], c=color, s=80)

        plt.title("Rectángulos de tinta detectados en HSI PNG")
        plt.axis("off")
        plt.show()

        # Plot máscaras por color
        for ink, mask in hsi_meta["debug_masks"].items():
            if np.count_nonzero(mask) == 0:
                continue

            plt.figure(figsize=(8, 5))
            plt.imshow(mask, cmap="gray")
            plt.title(f"Máscara detectada para tinta: {ink}")
            plt.axis("off")
            plt.show()

    return result

In [ ]:
he_xml_path = r"Datos\SB013\SB013\H&E_python_EDU_creado\H&E.xml"
hsi_ink_png_path = r"Datos\SB013\SB013\SB013_ink_annotation.png"

annotations = extract_he_hsi_ink_annotations(
    he_xml_path=he_xml_path,
    hsi_ink_png_path=hsi_ink_png_path,
    keep_inks=("green", "red", "blue"),
    show_debug=True,
)

def plot_he_mrxs_with_xml_ink_annotations(
    slide_path,
    annotations,
    max_thumb_size=1800,
    show_bbox=True,
    show_centroid=True,
    linewidth=3,
):
    """
    Abre la imagen H&E .mrxs como thumbnail y pinta encima las anotaciones
    de tinta extraídas del XML.

    Sirve para validar que:
    - las coordenadas XML están bien interpretadas
    - las tintas caen donde deben
    - el sistema source/destination elegido es correcto

    Parámetros
    ----------
    slide_path : str or Path
        Ruta al .mrxs.

    annotations : dict
        Salida de extract_he_hsi_ink_annotations(...)

    max_thumb_size : int
        Tamaño máximo del thumbnail en píxeles.

    Devuelve
    --------
    out : dict
        thumbnail_rgb, scale_x, scale_y, thumb_size, slide_size.
    """

    slide_path = Path(slide_path)

    slide = openslide.OpenSlide(str(slide_path))

    slide_w, slide_h = slide.dimensions

    # Crear thumbnail manteniendo proporción
    scale = max_thumb_size / max(slide_w, slide_h)
    thumb_w = int(slide_w * scale)
    thumb_h = int(slide_h * scale)

    thumb_pil = slide.get_thumbnail((thumb_w, thumb_h)).convert("RGB")
    thumb_rgb = np.array(thumb_pil)

    # En teoría, si las coordenadas del XML están en level 0/source,
    # el factor es thumbnail / slide dimensions.
    scale_x = thumb_w / slide_w
    scale_y = thumb_h / slide_h

    print("\n==============================")
    print("H&E MRXS THUMBNAIL")
    print("==============================")
    print("slide dimensions:", (slide_w, slide_h))
    print("thumbnail size:", (thumb_w, thumb_h))
    print("scale_x:", scale_x)
    print("scale_y:", scale_y)

    plt.figure(figsize=(12, 10))
    plt.imshow(thumb_rgb)

    for ann in annotations["HE"]:
        ink = ann["ink"]
        pts = ann["points"].copy()

        # Pasar de coordenadas XML/source a coordenadas thumbnail
        pts_thumb = pts.copy()
        pts_thumb[:, 0] *= scale_x
        pts_thumb[:, 1] *= scale_y

        cx, cy = ann["centroid"]
        cx_thumb = cx * scale_x
        cy_thumb = cy * scale_y

        x1, y1, x2, y2 = ann["bbox"]
        x1_t = x1 * scale_x
        y1_t = y1 * scale_y
        x2_t = x2 * scale_x
        y2_t = y2 * scale_y

        if ink == "green":
            color = "lime"
        elif ink == "red":
            color = "red"
        elif ink == "blue":
            color = "blue"
        else:
            color = "yellow"

        poly = Polygon(
            pts_thumb,
            closed=True,
            fill=False,
            edgecolor=color,
            linewidth=linewidth,
            label=f"HE {ink}"
        )
        plt.gca().add_patch(poly)

        if show_bbox:
            plt.gca().add_patch(
                Rectangle(
                    (x1_t, y1_t),
                    x2_t - x1_t,
                    y2_t - y1_t,
                    fill=False,
                    edgecolor=color,
                    linewidth=1.5,
                    linestyle="--",
                )
            )

        if show_centroid:
            plt.scatter(
                [cx_thumb],
                [cy_thumb],
                c=color,
                s=80,
                edgecolors="black",
                linewidths=1,
                zorder=10,
            )

        plt.text(
            cx_thumb + 8,
            cy_thumb + 8,
            ink,
            color="black",
            fontsize=11,
            bbox=dict(facecolor=color, alpha=0.8, edgecolor="black"),
        )

    plt.title("H&E .mrxs thumbnail con anotaciones de tinta del XML")
    plt.axis("off")
    plt.legend()
    plt.show()

    slide.close()

    return {
        "thumbnail_rgb": thumb_rgb,
        "slide_size": (slide_w, slide_h),
        "thumb_size": (thumb_w, thumb_h),
        "scale_x": scale_x,
        "scale_y": scale_y,
    }

In [ ]:
he_inks = annotations["HE"]
hsi_inks = annotations["HSI"]
common_inks = annotations["common_inks"]

print(common_inks)

In [ ]:
for ann in annotations["HE"]:
    print("HE", ann["ink"], ann["centroid"], ann["bbox"])

for ann in annotations["HSI"]:
    print("HSI", ann["ink"], ann["centroid"], ann["bbox"])

In [ ]:
slide_path = r"Datos\SB013\SB013\H&E_python_EDU_creado\SB013.mrxs"

he_debug = plot_he_mrxs_with_xml_ink_annotations(
    slide_path=slide_path,
    annotations=annotations,
    max_thumb_size=1800,
)

## Datos para entender

1. Diagnóstico H&E

In [ ]:
import openslide
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

slide_path = Path(r"Datos\SB013\SB013\H&E_python_EDU_creado\SB013.mrxs")

# ------------------------------------------------------------
# A) Ver niveles del .mrxs
# ------------------------------------------------------------
slide = openslide.OpenSlide(str(slide_path))

print("====================================")
print("NIVELES DEL .MRXS")
print("====================================")
print("slide.dimensions:", slide.dimensions)

for i, (w, h) in enumerate(slide.level_dimensions):
    print(f"level {i}: {w} x {h}")

slide.close()


# ------------------------------------------------------------
# B) Ejecutar tu función H&E con máscara + info
# ------------------------------------------------------------
he_tissue_rgb, he_mask, he_info = extract_he_tissue_contour_image(
    slide_path=slide_path,
    max_dim=1800,
    show_original=True,
    show_result=True,
    debug=True,
    return_mask=True,
    return_info=True
)

print("\n====================================")
print("H&E INFO")
print("====================================")
for k, v in he_info.items():
    print(k, ":", v)

print("\nhe_tissue_rgb.shape:", he_tissue_rgb.shape)
print("he_mask.shape:", he_mask.shape)
print("he_mask unique values:", np.unique(he_mask))
print("he_mask nonzero pixels:", np.count_nonzero(he_mask))


# ------------------------------------------------------------
# C) Plot simple: imagen H&E + máscara encima
# ------------------------------------------------------------
plt.figure(figsize=(8, 8))
plt.imshow(he_tissue_rgb)
plt.imshow(he_mask > 0, cmap="gray", alpha=0.35)
plt.title("H&E: tejido + máscara")
plt.axis("off")
plt.show()

2. Diagnóstico HSI

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from spectral import open_image

hdr_path = r"Datos\SB013\SB013\SB013_001\SB013_nrm.hdr"
hsi_ink_png_path = r"Datos\SB013\SB013\SB013_ink_annotation.png"


# ------------------------------------------------------------
# A) Ver tamaño real del cubo HSI
# ------------------------------------------------------------
img_hsi = open_image(hdr_path)
cube_shape = img_hsi.shape

print("\n====================================")
print("HSI HDR INFO")
print("====================================")
print("HSI cube shape:", cube_shape)
print("metadata samples:", img_hsi.metadata.get("samples"))
print("metadata lines:", img_hsi.metadata.get("lines"))
print("metadata bands:", img_hsi.metadata.get("bands"))


# ------------------------------------------------------------
# B) Ejecutar tu función HSI con máscara
# ------------------------------------------------------------
hsi_specimen_rgb, hsi_mask = extract_specimen_only_from_hsi_path(
    hdr_path=hdr_path,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=True
)

print("\n====================================")
print("HSI SPECIMEN INFO")
print("====================================")
print("hsi_specimen_rgb.shape:", hsi_specimen_rgb.shape)
print("hsi_mask.shape:", hsi_mask.shape)
print("hsi_mask unique values:", np.unique(hsi_mask))
print("hsi_mask nonzero pixels:", np.count_nonzero(hsi_mask))


# ------------------------------------------------------------
# C) Ver tamaño del PNG anotado
# ------------------------------------------------------------
hsi_png_bgr = cv2.imread(hsi_ink_png_path, cv2.IMREAD_COLOR)
hsi_png_rgb = cv2.cvtColor(hsi_png_bgr, cv2.COLOR_BGR2RGB)

print("\n====================================")
print("HSI PNG ANOTADO INFO")
print("====================================")
print("hsi_png_rgb.shape:", hsi_png_rgb.shape)

png_h, png_w = hsi_png_rgb.shape[:2]
hsi_h, hsi_w = hsi_specimen_rgb.shape[:2]

print("PNG size W,H:", (png_w, png_h))
print("HSI RGB size W,H:", (hsi_w, hsi_h))
print("scale PNG -> HSI:")
print("  scale_x =", hsi_w / png_w)
print("  scale_y =", hsi_h / png_h)


# ------------------------------------------------------------
# D) Plot HSI + máscara
# ------------------------------------------------------------
plt.figure(figsize=(8, 8))
plt.imshow(hsi_specimen_rgb)
plt.imshow(hsi_mask > 0, cmap="gray", alpha=0.35)
plt.title("HSI: espécimen + máscara")
plt.axis("off")
plt.show()


# ------------------------------------------------------------
# E) Plot PNG anotado, para comparar visualmente
# ------------------------------------------------------------
plt.figure(figsize=(8, 8))
plt.imshow(hsi_png_rgb)
plt.title("HSI PNG anotado")
plt.axis("off")
plt.show()

3. Diagnóstico de anotaciones ya extraídas

In [ ]:
print("\n====================================")
print("ANNOTATIONS META")
print("====================================")
print(annotations["meta"])

print("\n====================================")
print("HE ANNOTATIONS")
print("====================================")
for a in annotations["HE"]:
    print(
        a["ink"],
        "| raw:", a["raw_name"],
        "| centroid:", a["centroid"],
        "| bbox:", a["bbox"]
    )

print("\n====================================")
print("HSI ANNOTATIONS")
print("====================================")
for a in annotations["HSI"]:
    print(
        a["ink"],
        "| centroid:", a["centroid"],
        "| bbox:", a["bbox"]
    )

print("\ncommon_inks:", annotations["common_inks"])

En H&E hay que escalar coordenadas del XML a la imagen pequeña de 515 x 1170.
En HSI no hay que escalar porque PNG y HSI tienen el mismo tamaño.

## Pasar anotaciones a coordenadas relativas

Ahora vamos al siguiente paso: pasar las anotaciones a coordenadas relativas del espécimen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle


def get_ink_color(ink):
    if ink == "green":
        return "lime"
    if ink == "red":
        return "red"
    if ink == "blue":
        return "blue"
    return "yellow"


def bbox_from_mask(mask, margin=20):
    """
    Calcula el bounding box del espécimen a partir de su máscara.
    """
    mask_bool = mask > 0

    ys, xs = np.where(mask_bool)

    if len(xs) == 0:
        raise ValueError("La máscara está vacía.")

    h, w = mask_bool.shape

    x1 = max(int(xs.min()) - margin, 0)
    y1 = max(int(ys.min()) - margin, 0)
    x2 = min(int(xs.max()) + margin, w - 1)
    y2 = min(int(ys.max()) + margin, h - 1)

    return x1, y1, x2, y2


def transform_annotations_to_specimen_space(
    image_rgb,
    mask_specimen,
    annotations_list,
    modality_name,
    annotation_scale=(1.0, 1.0),
    bbox_margin=20,
    show_debug=True,
):
    """
    Pasa anotaciones a coordenadas relativas al espécimen.

    image_rgb:
        Imagen donde queremos pintar.
        H&E: he_tissue_rgb
        HSI: hsi_specimen_rgb

    mask_specimen:
        Máscara del espécimen.
        H&E: he_mask
        HSI: hsi_mask

    annotations_list:
        annotations["HE"] o annotations["HSI"]

    annotation_scale:
        Escala para pasar coordenadas originales a coordenadas de image_rgb.
    """

    image_rgb = np.asarray(image_rgb)
    mask = mask_specimen > 0

    img_h, img_w = image_rgb.shape[:2]
    mask_h, mask_w = mask.shape[:2]

    if (img_h, img_w) != (mask_h, mask_w):
        raise ValueError(
            f"{modality_name}: imagen y máscara no tienen el mismo tamaño.\n"
            f"image_rgb: {image_rgb.shape[:2]}\n"
            f"mask:      {mask.shape[:2]}"
        )

    # ------------------------------------------------------------
    # 1. Bounding box del espécimen
    # ------------------------------------------------------------
    x1, y1, x2, y2 = bbox_from_mask(mask, margin=bbox_margin)

    crop_rgb = image_rgb[y1:y2 + 1, x1:x2 + 1].copy()
    crop_mask = mask[y1:y2 + 1, x1:x2 + 1].copy()

    crop_h, crop_w = crop_rgb.shape[:2]

    sx, sy = annotation_scale

    transformed_annotations = []

    # ------------------------------------------------------------
    # 2. Transformar cada anotación
    # ------------------------------------------------------------
    for ann in annotations_list:
        pts_original = np.asarray(ann["points"], dtype=float)
        centroid_original = np.asarray(ann["centroid"], dtype=float)

        # A) Coordenadas originales -> coordenadas de image_rgb
        pts_img = pts_original.copy()
        pts_img[:, 0] *= sx
        pts_img[:, 1] *= sy

        centroid_img = centroid_original.copy()
        centroid_img[0] *= sx
        centroid_img[1] *= sy

        # B) Coordenadas de image_rgb -> coordenadas locales al crop del espécimen
        pts_local = pts_img.copy()
        pts_local[:, 0] -= x1
        pts_local[:, 1] -= y1

        centroid_local = centroid_img.copy()
        centroid_local[0] -= x1
        centroid_local[1] -= y1

        # C) Coordenadas locales -> coordenadas normalizadas dentro del espécimen
        pts_norm = pts_local.copy()
        pts_norm[:, 0] /= crop_w
        pts_norm[:, 1] /= crop_h

        centroid_norm = np.array(
            [
                centroid_local[0] / crop_w,
                centroid_local[1] / crop_h,
            ],
            dtype=float
        )

        # D) Comprobar si el centroide cae dentro de la máscara
        cx = int(round(centroid_img[0]))
        cy = int(round(centroid_img[1]))

        if 0 <= cx < img_w and 0 <= cy < img_h:
            inside_mask = bool(mask[cy, cx])
        else:
            inside_mask = False

        ann_new = {
            **ann,
            "points_img": pts_img,
            "centroid_img": centroid_img,
            "points_local": pts_local,
            "centroid_local": centroid_local,
            "points_norm_specimen": pts_norm,
            "centroid_norm_specimen": centroid_norm,
            "centroid_inside_specimen": inside_mask,
        }

        transformed_annotations.append(ann_new)

    result = {
        "modality": modality_name,
        "image_rgb": image_rgb,
        "mask": mask,
        "bbox": (x1, y1, x2, y2),
        "crop_rgb": crop_rgb,
        "crop_mask": crop_mask,
        "crop_size": (crop_w, crop_h),
        "image_size": (img_w, img_h),
        "annotation_scale": annotation_scale,
        "annotations": transformed_annotations,
    }

    # ------------------------------------------------------------
    # 3. Plots de comprobación
    # ------------------------------------------------------------
    if show_debug:
        print("\n====================================")
        print(f"{modality_name} - coordenadas relativas al espécimen")
        print("====================================")
        print("image_size:", (img_w, img_h))
        print("specimen_bbox:", (x1, y1, x2, y2))
        print("crop_size:", (crop_w, crop_h))
        print("annotation_scale:", annotation_scale)

        for a in transformed_annotations:
            print(
                f"{modality_name} {a['ink']:>5s} | "
                f"centroid_img={np.round(a['centroid_img'], 2)} | "
                f"centroid_local={np.round(a['centroid_local'], 2)} | "
                f"centroid_norm={np.round(a['centroid_norm_specimen'], 4)} | "
                f"inside_mask={a['centroid_inside_specimen']}"
            )

        # Imagen completa usada + máscara + bbox + anotaciones
        plt.figure(figsize=(8, 8))
        plt.imshow(image_rgb)
        plt.imshow(mask, cmap="gray", alpha=0.25)

        plt.gca().add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                edgecolor="yellow",
                linewidth=3,
                label="bbox espécimen"
            )
        )

        for a in transformed_annotations:
            color = get_ink_color(a["ink"])

            plt.gca().add_patch(
                Polygon(
                    a["points_img"],
                    closed=True,
                    fill=False,
                    edgecolor=color,
                    linewidth=3,
                    label=f"{modality_name} {a['ink']}"
                )
            )

            cx, cy = a["centroid_img"]
            plt.scatter(
                [cx],
                [cy],
                c=color,
                s=80,
                edgecolors="black",
                zorder=10
            )

        plt.title(f"{modality_name}: imagen completa + máscara + anotaciones")
        plt.axis("off")
        plt.legend()
        plt.show()

        # Crop del espécimen + anotaciones locales
        plt.figure(figsize=(8, 8))
        plt.imshow(crop_rgb)
        plt.imshow(crop_mask, cmap="gray", alpha=0.25)

        for a in transformed_annotations:
            color = get_ink_color(a["ink"])

            plt.gca().add_patch(
                Polygon(
                    a["points_local"],
                    closed=True,
                    fill=False,
                    edgecolor=color,
                    linewidth=3
                )
            )

            cx, cy = a["centroid_local"]

            plt.scatter(
                [cx],
                [cy],
                c=color,
                s=80,
                edgecolors="black",
                zorder=10
            )

            plt.text(
                cx + 5,
                cy + 5,
                f"{a['ink']}\n{np.round(a['centroid_norm_specimen'], 3)}",
                color="black",
                fontsize=9,
                bbox=dict(facecolor=color, alpha=0.8, edgecolor="black")
            )

        plt.title(f"{modality_name}: crop espécimen + coordenadas normalizadas")
        plt.axis("off")
        plt.show()

    return result

1. Aplicarlo a H&E

In [ ]:
he_tissue_rgb
he_mask
he_info
annotations["HE"]

In [ ]:
# Tamaño del sistema de coordenadas del XML
xml_w, xml_h = annotations["meta"]["HE"]["coord_size_used"]

# Tamaño de la imagen H&E pequeña que realmente estamos usando
he_w, he_h = he_info["read_dimensions"]

# Escala XML -> H&E pequeña
he_scale_x = he_w / xml_w
he_scale_y = he_h / xml_h

print("H&E scale:")
print("he_scale_x =", he_scale_x)
print("he_scale_y =", he_scale_y)

he_space = transform_annotations_to_specimen_space(
    image_rgb=he_tissue_rgb,
    mask_specimen=he_mask,
    annotations_list=annotations["HE"],
    modality_name="HE",
    annotation_scale=(he_scale_x, he_scale_y),
    bbox_margin=20,
    show_debug=True,
)

2. Aplicarlo a HSI

In [ ]:
hsi_specimen_rgb
hsi_mask
annotations["HSI"]

In [ ]:
# Tamaño del PNG anotado
png_w, png_h = annotations["meta"]["HSI"]["image_size"]

# Tamaño de la imagen HSI usada
hsi_h, hsi_w = hsi_specimen_rgb.shape[:2]

# Escala PNG -> HSI
hsi_scale_x = hsi_w / png_w
hsi_scale_y = hsi_h / png_h

print("HSI scale:")
print("hsi_scale_x =", hsi_scale_x)
print("hsi_scale_y =", hsi_scale_y)

hsi_space = transform_annotations_to_specimen_space(
    image_rgb=hsi_specimen_rgb,
    mask_specimen=hsi_mask,
    annotations_list=annotations["HSI"],
    modality_name="HSI",
    annotation_scale=(hsi_scale_x, hsi_scale_y),
    bbox_margin=20,
    show_debug=True,
)

3. Comparar tinta común

In [ ]:
def compare_common_inks(he_space, hsi_space):
    he_by_ink = {
        a["ink"]: a
        for a in he_space["annotations"]
    }

    hsi_by_ink = {
        a["ink"]: a
        for a in hsi_space["annotations"]
    }

    common_inks = sorted(set(he_by_ink.keys()) & set(hsi_by_ink.keys()))

    print("\n====================================")
    print("Comparación de tintas comunes")
    print("====================================")
    print("common_inks:", common_inks)

    rows = []

    for ink in common_inks:
        he_p = he_by_ink[ink]["centroid_norm_specimen"]
        hsi_p = hsi_by_ink[ink]["centroid_norm_specimen"]

        diff = hsi_p - he_p
        dist = np.linalg.norm(diff)

        print(f"\nINK: {ink}")
        print("HE  norm:", np.round(he_p, 4))
        print("HSI norm:", np.round(hsi_p, 4))
        print("diff    :", np.round(diff, 4))
        print("dist    :", round(float(dist), 4))

        rows.append((ink, he_p, hsi_p, diff, dist))

    plt.figure(figsize=(6, 6))

    for ink, he_p, hsi_p, diff, dist in rows:
        color = get_ink_color(ink)

        plt.scatter(
            he_p[0],
            he_p[1],
            c=color,
            s=130,
            marker="o",
            edgecolors="black",
            label=f"HE {ink}"
        )

        plt.scatter(
            hsi_p[0],
            hsi_p[1],
            c=color,
            s=130,
            marker="s",
            edgecolors="black",
            label=f"HSI {ink}"
        )

        plt.plot(
            [he_p[0], hsi_p[0]],
            [he_p[1], hsi_p[1]],
            linestyle="--",
            color="black",
            alpha=0.6
        )

    plt.xlim(0, 1)
    plt.ylim(1, 0)
    plt.grid(True)
    plt.xlabel("x normalizado dentro del espécimen")
    plt.ylabel("y normalizado dentro del espécimen")
    plt.title("Tintas comunes en coordenadas normalizadas")
    plt.legend()
    plt.show()

    return rows


ink_comparison = compare_common_inks(
    he_space=he_space,
    hsi_space=hsi_space
)

## probar orientaciones candidatas de la H&E respecto a la HSI

In [ ]:
he_space
hsi_space

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# 1. Transformaciones geométricas de imagen y puntos
# ============================================================

def apply_orientation_image(img, k_rot90=0, flip_h=False):
    """
    Aplica orientación a una imagen/máscara.

    Orden:
    1) flip horizontal opcional
    2) rotación k_rot90 veces 90º antihorario

    k_rot90:
        0 -> 0 grados
        1 -> 90 grados antihorario
        2 -> 180 grados
        3 -> 270 grados antihorario
    """
    out = img.copy()

    if flip_h:
        out = np.fliplr(out)

    out = np.rot90(out, k=k_rot90)

    return out.copy()


def apply_orientation_points(points, width, height, k_rot90=0, flip_h=False):
    """
    Aplica exactamente la misma orientación a puntos 2D.

    points:
        array Nx2 con columnas [x, y]

    width, height:
        tamaño de la imagen original donde viven esos puntos.

    Devuelve:
        points_transformed, new_width, new_height
    """
    pts = np.asarray(points, dtype=float).copy()

    current_w = width
    current_h = height

    # 1) Flip horizontal
    if flip_h:
        pts[:, 0] = (current_w - 1) - pts[:, 0]

    # 2) Rotaciones 90º antihorarias
    for _ in range(k_rot90):
        x = pts[:, 0].copy()
        y = pts[:, 1].copy()

        # np.rot90 antihorario:
        # x' = y
        # y' = width - 1 - x
        pts[:, 0] = y
        pts[:, 1] = (current_w - 1) - x

        current_w, current_h = current_h, current_w

    return pts, current_w, current_h


def get_orientation_candidates():
    """
    Las 8 orientaciones candidatas.
    """
    candidates = []

    for flip_h in [False, True]:
        for k in [0, 1, 2, 3]:
            if not flip_h:
                name = f"rot{k * 90}_ccw"
            else:
                name = f"flipH_rot{k * 90}_ccw"

            candidates.append(
                {
                    "name": name,
                    "k_rot90": k,
                    "flip_h": flip_h,
                }
            )

    return candidates


# ============================================================
# 2. Distancia de forma entre máscaras
# ============================================================

def binary_edge(mask):
    """
    Extrae borde de una máscara binaria usando gradiente morfológico.
    """
    mask_u8 = (mask > 0).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    dil = cv2.dilate(mask_u8, kernel, iterations=1)
    ero = cv2.erode(mask_u8, kernel, iterations=1)

    edge = (dil - ero) > 0

    return edge


def symmetric_chamfer_distance(mask_a, mask_b):
    """
    Distancia aproximada entre contornos de dos máscaras.

    Devuelve distancia normalizada por la diagonal de la imagen.
    Menor = mejor.
    """
    mask_a = mask_a > 0
    mask_b = mask_b > 0

    edge_a = binary_edge(mask_a)
    edge_b = binary_edge(mask_b)

    if edge_a.sum() < 5 or edge_b.sum() < 5:
        return np.inf

    h, w = mask_a.shape[:2]
    diag = np.sqrt(h ** 2 + w ** 2)

    # distanceTransform mide distancia a píxeles 0.
    # Ponemos 0 en el borde y 255 fuera.
    inv_b = (~edge_b).astype(np.uint8) * 255
    inv_a = (~edge_a).astype(np.uint8) * 255

    dist_to_b = cv2.distanceTransform(inv_b, cv2.DIST_L2, 3)
    dist_to_a = cv2.distanceTransform(inv_a, cv2.DIST_L2, 3)

    d_ab = dist_to_b[edge_a].mean()
    d_ba = dist_to_a[edge_b].mean()

    return float((d_ab + d_ba) / 2.0 / diag)


# ============================================================
# 3. Evaluar orientaciones
# ============================================================

def evaluate_he_orientations_against_hsi(
    he_space,
    hsi_space,
    ink_weight=0.80,
    shape_weight=0.20,
):
    """
    Prueba las 8 orientaciones de H&E contra HSI.

    Score:
        score = ink_weight * error_tintas + shape_weight * error_contorno

    Como solo tenemos una tinta común, el error de tinta pesa bastante.
    """

    he_img = he_space["crop_rgb"]
    he_mask = he_space["crop_mask"]
    hsi_img = hsi_space["crop_rgb"]
    hsi_mask = hsi_space["crop_mask"]

    he_h, he_w = he_img.shape[:2]
    hsi_h, hsi_w = hsi_img.shape[:2]

    he_by_ink = {a["ink"]: a for a in he_space["annotations"]}
    hsi_by_ink = {a["ink"]: a for a in hsi_space["annotations"]}

    common_inks = sorted(set(he_by_ink.keys()) & set(hsi_by_ink.keys()))

    if len(common_inks) == 0:
        print("AVISO: no hay tintas comunes. Se usará solo forma.")
    else:
        print("Tintas comunes usadas:", common_inks)

    results = []

    for cand in get_orientation_candidates():
        name = cand["name"]
        k = cand["k_rot90"]
        flip_h = cand["flip_h"]

        # --------------------------------------------------------
        # A) Orientar imagen y máscara H&E
        # --------------------------------------------------------
        he_img_oriented = apply_orientation_image(
            he_img,
            k_rot90=k,
            flip_h=flip_h,
        )

        he_mask_oriented = apply_orientation_image(
            he_mask.astype(np.uint8),
            k_rot90=k,
            flip_h=flip_h,
        ) > 0

        he_or_h, he_or_w = he_img_oriented.shape[:2]

        # --------------------------------------------------------
        # B) Orientar anotaciones H&E
        # --------------------------------------------------------
        oriented_annotations = {}

        for ink, ann in he_by_ink.items():
            pts_local = ann["points_local"]
            centroid_local = ann["centroid_local"].reshape(1, 2)

            pts_t, new_w, new_h = apply_orientation_points(
                pts_local,
                width=he_w,
                height=he_h,
                k_rot90=k,
                flip_h=flip_h,
            )

            centroid_t, _, _ = apply_orientation_points(
                centroid_local,
                width=he_w,
                height=he_h,
                k_rot90=k,
                flip_h=flip_h,
            )

            centroid_t = centroid_t[0]

            centroid_norm = np.array(
                [
                    centroid_t[0] / new_w,
                    centroid_t[1] / new_h,
                ],
                dtype=float,
            )

            oriented_annotations[ink] = {
                "points_local_oriented": pts_t,
                "centroid_local_oriented": centroid_t,
                "centroid_norm_oriented": centroid_norm,
            }

        # --------------------------------------------------------
        # C) Error de tintas
        # --------------------------------------------------------
        ink_errors = []

        for ink in common_inks:
            he_p = oriented_annotations[ink]["centroid_norm_oriented"]
            hsi_p = hsi_by_ink[ink]["centroid_norm_specimen"]

            d = np.linalg.norm(he_p - hsi_p)
            ink_errors.append(d)

        if len(ink_errors) > 0:
            ink_error = float(np.mean(ink_errors))
        else:
            ink_error = 0.0

        # --------------------------------------------------------
        # D) Error de forma
        #    Redimensionamos la máscara H&E orientada al tamaño del crop HSI.
        # --------------------------------------------------------
        he_mask_resized = cv2.resize(
            he_mask_oriented.astype(np.uint8),
            (hsi_w, hsi_h),
            interpolation=cv2.INTER_NEAREST,
        ) > 0

        shape_error = symmetric_chamfer_distance(
            he_mask_resized,
            hsi_mask,
        )

        # --------------------------------------------------------
        # E) Score total
        # --------------------------------------------------------
        total_score = ink_weight * ink_error + shape_weight * shape_error

        results.append(
            {
                "name": name,
                "k_rot90": k,
                "flip_h": flip_h,
                "score": total_score,
                "ink_error": ink_error,
                "shape_error": shape_error,
                "he_img_oriented": he_img_oriented,
                "he_mask_oriented": he_mask_oriented,
                "he_mask_resized_to_hsi": he_mask_resized,
                "oriented_annotations": oriented_annotations,
            }
        )

    results = sorted(results, key=lambda r: r["score"])

    print("\n====================================")
    print("Ranking de orientaciones")
    print("====================================")

    for i, r in enumerate(results):
        print(
            f"{i + 1:02d}. {r['name']:18s} | "
            f"score={r['score']:.4f} | "
            f"ink={r['ink_error']:.4f} | "
            f"shape={r['shape_error']:.4f}"
        )

        for ink in common_inks:
            he_p = r["oriented_annotations"][ink]["centroid_norm_oriented"]
            hsi_p = hsi_by_ink[ink]["centroid_norm_specimen"]

            print(
                f"      {ink}: HE_oriented={np.round(he_p, 4)} "
                f"HSI={np.round(hsi_p, 4)}"
            )

    return results


orientation_results = evaluate_he_orientations_against_hsi(
    he_space=he_space,
    hsi_space=hsi_space,
    ink_weight=0.80,
    shape_weight=0.20,
)

In [ ]:
def plot_orientation_result(result, hsi_space, common_inks=None):
    """
    Visualiza una orientación candidata.
    """

    if common_inks is None:
        he_inks = set(result["oriented_annotations"].keys())
        hsi_inks = set(a["ink"] for a in hsi_space["annotations"])
        common_inks = sorted(he_inks & hsi_inks)

    hsi_img = hsi_space["crop_rgb"]
    hsi_mask = hsi_space["crop_mask"]

    hsi_h, hsi_w = hsi_img.shape[:2]

    hsi_by_ink = {
        a["ink"]: a
        for a in hsi_space["annotations"]
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # ------------------------------------------------------------
    # 1. H&E orientada
    # ------------------------------------------------------------
    ax = axes[0]
    ax.imshow(result["he_img_oriented"])
    ax.imshow(result["he_mask_oriented"], cmap="gray", alpha=0.25)
    ax.set_title(f"H&E orientada\n{result['name']}")
    ax.axis("off")

    for ink in common_inks:
        if ink not in result["oriented_annotations"]:
            continue

        color = get_ink_color(ink)
        p = result["oriented_annotations"][ink]["centroid_local_oriented"]

        ax.scatter(
            [p[0]],
            [p[1]],
            c=color,
            s=90,
            edgecolors="black",
            zorder=10,
        )

        ax.text(
            p[0] + 5,
            p[1] + 5,
            f"HE {ink}",
            color="black",
            fontsize=9,
            bbox=dict(facecolor=color, alpha=0.8, edgecolor="black"),
        )

    # ------------------------------------------------------------
    # 2. HSI con punto HSI y punto H&E orientado proyectado
    # ------------------------------------------------------------
    ax = axes[1]
    ax.imshow(hsi_img)
    ax.imshow(hsi_mask, cmap="gray", alpha=0.25)
    ax.set_title("HSI crop\npuntos normalizados")
    ax.axis("off")

    for ink in common_inks:
        color = get_ink_color(ink)

        he_norm = result["oriented_annotations"][ink]["centroid_norm_oriented"]
        hsi_norm = hsi_by_ink[ink]["centroid_norm_specimen"]

        he_xy_on_hsi = np.array(
            [
                he_norm[0] * hsi_w,
                he_norm[1] * hsi_h,
            ]
        )

        hsi_xy = np.array(
            [
                hsi_norm[0] * hsi_w,
                hsi_norm[1] * hsi_h,
            ]
        )

        ax.scatter(
            [hsi_xy[0]],
            [hsi_xy[1]],
            c=color,
            marker="s",
            s=110,
            edgecolors="black",
            label=f"HSI {ink}",
            zorder=10,
        )

        ax.scatter(
            [he_xy_on_hsi[0]],
            [he_xy_on_hsi[1]],
            c=color,
            marker="o",
            s=110,
            edgecolors="black",
            label=f"HE {ink} orientado",
            zorder=10,
        )

        ax.plot(
            [hsi_xy[0], he_xy_on_hsi[0]],
            [hsi_xy[1], he_xy_on_hsi[1]],
            "--",
            color="black",
            linewidth=1.5,
        )

    ax.legend()

    # ------------------------------------------------------------
    # 3. Comparación de contornos
    # ------------------------------------------------------------
    ax = axes[2]
    ax.imshow(hsi_img)
    ax.contour(
        hsi_mask > 0,
        levels=[0.5],
        colors="yellow",
        linewidths=2,
    )
    ax.contour(
        result["he_mask_resized_to_hsi"] > 0,
        levels=[0.5],
        colors="cyan",
        linewidths=2,
    )

    ax.set_title(
        "Contornos sobre tamaño HSI\n"
        "amarillo=HSI | cian=H&E orientada"
    )
    ax.axis("off")

    plt.suptitle(
        f"{result['name']} | score={result['score']:.4f} | "
        f"ink={result['ink_error']:.4f} | shape={result['shape_error']:.4f}",
        fontsize=13,
    )

    plt.tight_layout()
    plt.show()


# Visualizar las 4 mejores
for r in orientation_results[:4]:
    plot_orientation_result(
        result=r,
        hsi_space=hsi_space,
    )

La H&E, para parecerse a la HSI, habría que girarla 90 grados antihorario.

## Crear una nueva variable con la H&E ya orientada

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def get_ink_color(ink):
    if ink == "green":
        return "lime"
    if ink == "red":
        return "red"
    if ink == "blue":
        return "blue"
    return "yellow"


def build_oriented_he_space_from_result(best_result, he_space):
    """
    Construye un nuevo objeto H&E ya orientado usando la mejor orientación
    encontrada en orientation_results.

    Usa:
        best_result["he_img_oriented"]
        best_result["he_mask_oriented"]
        best_result["oriented_annotations"]
    """

    he_img_oriented = best_result["he_img_oriented"]
    he_mask_oriented = best_result["he_mask_oriented"]

    h_or, w_or = he_img_oriented.shape[:2]

    oriented_annotations = []

    for ann in he_space["annotations"]:
        ink = ann["ink"]

        if ink not in best_result["oriented_annotations"]:
            continue

        ann_or = best_result["oriented_annotations"][ink]

        points_local_oriented = ann_or["points_local_oriented"]
        centroid_local_oriented = ann_or["centroid_local_oriented"]
        centroid_norm_oriented = ann_or["centroid_norm_oriented"]

        new_ann = {
            **ann,
            "points_local": points_local_oriented,
            "centroid_local": centroid_local_oriented,
            "points_norm_specimen": points_local_oriented / np.array([w_or, h_or]),
            "centroid_norm_specimen": centroid_norm_oriented,
            "orientation_name": best_result["name"],
        }

        oriented_annotations.append(new_ann)

    he_oriented_space = {
        "modality": "HE_oriented",
        "orientation_name": best_result["name"],
        "k_rot90": best_result["k_rot90"],
        "flip_h": best_result["flip_h"],
        "crop_rgb": he_img_oriented,
        "crop_mask": he_mask_oriented,
        "annotations": oriented_annotations,
        "crop_size": (w_or, h_or),
        "score": best_result["score"],
        "ink_error": best_result["ink_error"],
        "shape_error": best_result["shape_error"],
    }

    return he_oriented_space


# ============================================================
# Crear he_oriented_space
# ============================================================

best_orientation = orientation_results[0]

he_oriented_space = build_oriented_he_space_from_result(
    best_result=best_orientation,
    he_space=he_space,
)

print("====================================")
print("H&E ORIENTADA")
print("====================================")
print("Orientación elegida:", he_oriented_space["orientation_name"])
print("crop_size:", he_oriented_space["crop_size"])
print("score:", round(float(he_oriented_space["score"]), 4))
print("ink_error:", round(float(he_oriented_space["ink_error"]), 4))
print("shape_error:", round(float(he_oriented_space["shape_error"]), 4))

for ann in he_oriented_space["annotations"]:
    print(
        ann["ink"],
        "| centroid_local:",
        np.round(ann["centroid_local"], 2),
        "| centroid_norm:",
        np.round(ann["centroid_norm_specimen"], 4),
    )

In [ ]:
def plot_final_oriented_he_vs_hsi(he_oriented_space, hsi_space):
    """
    Visualiza H&E ya orientada frente a HSI.
    """

    he_img = he_oriented_space["crop_rgb"]
    he_mask = he_oriented_space["crop_mask"]

    hsi_img = hsi_space["crop_rgb"]
    hsi_mask = hsi_space["crop_mask"]

    he_by_ink = {a["ink"]: a for a in he_oriented_space["annotations"]}
    hsi_by_ink = {a["ink"]: a for a in hsi_space["annotations"]}

    common_inks = sorted(set(he_by_ink.keys()) & set(hsi_by_ink.keys()))

    hsi_h, hsi_w = hsi_img.shape[:2]

    plt.figure(figsize=(16, 6))

    # ============================================================
    # 1. H&E orientada
    # ============================================================

    plt.subplot(1, 3, 1)
    plt.imshow(he_img)
    plt.imshow(he_mask, cmap="gray", alpha=0.25)

    for ink, ann in he_by_ink.items():
        color = get_ink_color(ink)
        cx, cy = ann["centroid_local"]

        plt.scatter(
            [cx],
            [cy],
            c=color,
            s=90,
            edgecolors="black",
            zorder=10,
        )

        plt.text(
            cx + 5,
            cy + 5,
            f"HE {ink}",
            color="black",
            fontsize=9,
            bbox=dict(facecolor=color, alpha=0.8, edgecolor="black"),
        )

    plt.title(f"H&E orientada\n{he_oriented_space['orientation_name']}")
    plt.axis("off")

    # ============================================================
    # 2. HSI
    # ============================================================

    plt.subplot(1, 3, 2)
    plt.imshow(hsi_img)
    plt.imshow(hsi_mask, cmap="gray", alpha=0.25)

    for ink, ann in hsi_by_ink.items():
        color = get_ink_color(ink)
        cx, cy = ann["centroid_local"]

        plt.scatter(
            [cx],
            [cy],
            c=color,
            s=90,
            marker="s",
            edgecolors="black",
            zorder=10,
        )

        plt.text(
            cx + 5,
            cy + 5,
            f"HSI {ink}",
            color="black",
            fontsize=9,
            bbox=dict(facecolor=color, alpha=0.8, edgecolor="black"),
        )

    plt.title("HSI")
    plt.axis("off")

    # ============================================================
    # 3. Puntos normalizados sobre HSI
    # ============================================================

    plt.subplot(1, 3, 3)
    plt.imshow(hsi_img)
    plt.imshow(hsi_mask, cmap="gray", alpha=0.25)

    for ink in common_inks:
        color = get_ink_color(ink)

        he_norm = he_by_ink[ink]["centroid_norm_specimen"]
        hsi_norm = hsi_by_ink[ink]["centroid_norm_specimen"]

        he_xy_on_hsi = np.array(
            [
                he_norm[0] * hsi_w,
                he_norm[1] * hsi_h,
            ]
        )

        hsi_xy = np.array(
            [
                hsi_norm[0] * hsi_w,
                hsi_norm[1] * hsi_h,
            ]
        )

        plt.scatter(
            [hsi_xy[0]],
            [hsi_xy[1]],
            c=color,
            marker="s",
            s=120,
            edgecolors="black",
            label=f"HSI {ink}",
            zorder=10,
        )

        plt.scatter(
            [he_xy_on_hsi[0]],
            [he_xy_on_hsi[1]],
            c=color,
            marker="o",
            s=120,
            edgecolors="black",
            label=f"HE {ink} orientada",
            zorder=10,
        )

        plt.plot(
            [hsi_xy[0], he_xy_on_hsi[0]],
            [hsi_xy[1], he_xy_on_hsi[1]],
            "--",
            color="black",
            linewidth=1.5,
        )

    plt.title("Comparación de tintas normalizadas")
    plt.axis("off")
    plt.legend()

    plt.tight_layout()
    plt.show()


plot_final_oriented_he_vs_hsi(
    he_oriented_space=he_oriented_space,
    hsi_space=hsi_space,
)

## Prueba alineación

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def binary_bbox(mask):
    """
    Devuelve bbox de una máscara binaria como:
        x1, y1, x2, y2
    """
    mask = mask > 0
    ys, xs = np.where(mask)

    if len(xs) == 0:
        raise ValueError("La máscara está vacía.")

    x1 = int(xs.min())
    y1 = int(ys.min())
    x2 = int(xs.max())
    y2 = int(ys.max())

    return x1, y1, x2, y2


def bbox_center_and_size(bbox):
    """
    Devuelve centro y tamaño de un bbox.
    """
    x1, y1, x2, y2 = bbox

    w = x2 - x1 + 1
    h = y2 - y1 + 1

    cx = x1 + w / 2.0
    cy = y1 + h / 2.0

    return np.array([cx, cy], dtype=float), np.array([w, h], dtype=float)


def transform_points_affine(points, M):
    """
    Aplica una matriz affine 2x3 a puntos Nx2.
    """
    pts = np.asarray(points, dtype=float)
    pts_h = np.concatenate(
        [pts, np.ones((len(pts), 1), dtype=float)],
        axis=1
    )

    pts_t = pts_h @ M.T
    return pts_t


def build_initial_affine_from_masks(
    he_mask,
    hsi_mask,
    use_anisotropic_scale=True,
):
    """
    Construye una transformación affine inicial H&E -> HSI.

    Hace:
    - calcula bbox de H&E orientada
    - calcula bbox de HSI
    - escala H&E para aproximar el tamaño del bbox HSI
    - traslada H&E para centrarla en HSI

    use_anisotropic_scale=True:
        usa sx y sy distintos.
        Esto deforma ligeramente, pero suele ayudar como primera aproximación.

    use_anisotropic_scale=False:
        usa una escala única.
        Es más conservador.
    """

    he_mask = he_mask > 0
    hsi_mask = hsi_mask > 0

    he_bbox = binary_bbox(he_mask)
    hsi_bbox = binary_bbox(hsi_mask)

    he_center, he_size = bbox_center_and_size(he_bbox)
    hsi_center, hsi_size = bbox_center_and_size(hsi_bbox)

    sx = hsi_size[0] / he_size[0]
    sy = hsi_size[1] / he_size[1]

    if not use_anisotropic_scale:
        s = 0.5 * (sx + sy)
        sx = s
        sy = s

    tx = hsi_center[0] - sx * he_center[0]
    ty = hsi_center[1] - sy * he_center[1]

    M = np.array(
        [
            [sx, 0.0, tx],
            [0.0, sy, ty],
        ],
        dtype=np.float32
    )

    info = {
        "he_bbox": he_bbox,
        "hsi_bbox": hsi_bbox,
        "he_center": he_center,
        "hsi_center": hsi_center,
        "he_size": he_size,
        "hsi_size": hsi_size,
        "sx": sx,
        "sy": sy,
        "tx": tx,
        "ty": ty,
        "M": M,
    }

    return M, info


def initial_align_he_to_hsi(
    he_oriented_space,
    hsi_space,
    use_anisotropic_scale=True,
    show_debug=True,
):
    """
    Aplica alineación inicial H&E orientada -> HSI.

    Devuelve:
        aligned
    """

    he_rgb = he_oriented_space["crop_rgb"]
    he_mask = he_oriented_space["crop_mask"] > 0

    hsi_rgb = hsi_space["crop_rgb"]
    hsi_mask = hsi_space["crop_mask"] > 0

    hsi_h, hsi_w = hsi_rgb.shape[:2]

    # ------------------------------------------------------------
    # 1. Matriz affine inicial
    # ------------------------------------------------------------
    M, info = build_initial_affine_from_masks(
        he_mask=he_mask,
        hsi_mask=hsi_mask,
        use_anisotropic_scale=use_anisotropic_scale,
    )

    # ------------------------------------------------------------
    # 2. Warpear imagen H&E y máscara H&E al tamaño de HSI
    # ------------------------------------------------------------
    he_rgb_warped = cv2.warpAffine(
        he_rgb,
        M,
        (hsi_w, hsi_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )

    he_mask_warped_u8 = cv2.warpAffine(
        (he_mask.astype(np.uint8) * 255),
        M,
        (hsi_w, hsi_h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )

    he_mask_warped = he_mask_warped_u8 > 0

    # ------------------------------------------------------------
    # 3. Transformar anotaciones H&E al sistema HSI crop
    # ------------------------------------------------------------
    warped_annotations = []

    for ann in he_oriented_space["annotations"]:
        pts = np.asarray(ann["points_local"], dtype=float)
        centroid = np.asarray(ann["centroid_local"], dtype=float).reshape(1, 2)

        pts_warped = transform_points_affine(pts, M)
        centroid_warped = transform_points_affine(centroid, M)[0]

        centroid_norm_hsi = np.array(
            [
                centroid_warped[0] / hsi_w,
                centroid_warped[1] / hsi_h,
            ],
            dtype=float
        )

        warped_ann = {
            **ann,
            "points_hsi_crop": pts_warped,
            "centroid_hsi_crop": centroid_warped,
            "centroid_norm_hsi_crop": centroid_norm_hsi,
        }

        warped_annotations.append(warped_ann)

    aligned = {
        "M_he_to_hsi_initial": M,
        "info": info,
        "he_rgb_warped": he_rgb_warped,
        "he_mask_warped": he_mask_warped,
        "he_annotations_warped": warped_annotations,
        "hsi_rgb": hsi_rgb,
        "hsi_mask": hsi_mask,
        "use_anisotropic_scale": use_anisotropic_scale,
    }

    # ------------------------------------------------------------
    # 4. Debug prints y plots
    # ------------------------------------------------------------
    if show_debug:
        print("\n====================================")
        print("ALINEACIÓN INICIAL H&E -> HSI")
        print("====================================")
        print("use_anisotropic_scale:", use_anisotropic_scale)
        print("HE bbox:", info["he_bbox"])
        print("HSI bbox:", info["hsi_bbox"])
        print("HE center:", np.round(info["he_center"], 2))
        print("HSI center:", np.round(info["hsi_center"], 2))
        print("HE size:", np.round(info["he_size"], 2))
        print("HSI size:", np.round(info["hsi_size"], 2))
        print("sx:", round(float(info["sx"]), 4))
        print("sy:", round(float(info["sy"]), 4))
        print("tx:", round(float(info["tx"]), 2))
        print("ty:", round(float(info["ty"]), 2))
        print("M:\n", M)

        # Comparación de tinta común tras affine inicial
        he_by_ink = {
            a["ink"]: a
            for a in warped_annotations
        }

        hsi_by_ink = {
            a["ink"]: a
            for a in hsi_space["annotations"]
        }

        common_inks = sorted(set(he_by_ink.keys()) & set(hsi_by_ink.keys()))

        print("\nTintas comunes tras affine inicial:", common_inks)

        for ink in common_inks:
            he_p = he_by_ink[ink]["centroid_hsi_crop"]
            hsi_p = hsi_by_ink[ink]["centroid_local"]

            dist_px = np.linalg.norm(he_p - hsi_p)
            diag = np.sqrt(hsi_w ** 2 + hsi_h ** 2)
            dist_norm = dist_px / diag

            print(f"\nINK {ink}")
            print("  HE warped:", np.round(he_p, 2))
            print("  HSI      :", np.round(hsi_p, 2))
            print("  dist_px  :", round(float(dist_px), 2))
            print("  dist_norm:", round(float(dist_norm), 4))

        # --------------------------------------------------------
        # Plot overlay
        # --------------------------------------------------------
        overlay = hsi_rgb.copy().astype(np.float32)
        he_warp = he_rgb_warped.astype(np.float32)

        alpha = 0.45
        m = he_mask_warped

        overlay[m] = (1 - alpha) * overlay[m] + alpha * he_warp[m]
        overlay = np.clip(overlay, 0, 255).astype(np.uint8)

        fig, axes = plt.subplots(1, 4, figsize=(22, 6))

        # 1. HSI
        axes[0].imshow(hsi_rgb)
        axes[0].imshow(hsi_mask, cmap="gray", alpha=0.25)
        axes[0].set_title("HSI crop")
        axes[0].axis("off")

        # 2. H&E warpeada
        axes[1].imshow(he_rgb_warped)
        axes[1].imshow(he_mask_warped, cmap="gray", alpha=0.25)
        axes[1].set_title("H&E orientada + affine inicial")
        axes[1].axis("off")

        # 3. Overlay RGB
        axes[2].imshow(overlay)
        axes[2].set_title("Overlay HSI + H&E")
        axes[2].axis("off")

        # 4. Contornos + tintas
        axes[3].imshow(hsi_rgb)
        axes[3].contour(
            hsi_mask,
            levels=[0.5],
            colors="yellow",
            linewidths=2,
        )
        axes[3].contour(
            he_mask_warped,
            levels=[0.5],
            colors="cyan",
            linewidths=2,
        )

        for ann in warped_annotations:
            color = get_ink_color(ann["ink"])
            cx, cy = ann["centroid_hsi_crop"]

            axes[3].scatter(
                [cx],
                [cy],
                c=color,
                s=90,
                marker="o",
                edgecolors="black",
                zorder=10,
            )

            axes[3].text(
                cx + 5,
                cy + 5,
                f"HE {ann['ink']}",
                color="black",
                fontsize=8,
                bbox=dict(facecolor=color, alpha=0.8, edgecolor="black"),
            )

        for ann in hsi_space["annotations"]:
            color = get_ink_color(ann["ink"])
            cx, cy = ann["centroid_local"]

            axes[3].scatter(
                [cx],
                [cy],
                c=color,
                s=100,
                marker="s",
                edgecolors="black",
                zorder=10,
            )

            axes[3].text(
                cx + 5,
                cy + 5,
                f"HSI {ann['ink']}",
                color="black",
                fontsize=8,
                bbox=dict(facecolor=color, alpha=0.8, edgecolor="black"),
            )

        axes[3].set_title("Contornos + tintas\namarillo=HSI, cian=H&E")
        axes[3].axis("off")

        plt.tight_layout()
        plt.show()

    return aligned

In [ ]:
initial_alignment = initial_align_he_to_hsi(
    he_oriented_space=he_oriented_space,
    hsi_space=hsi_space,
    use_anisotropic_scale=True,
    show_debug=True,
)

Con escala uniforme:
La escala uniforme significa que escalas la imagen igual en X y en Y. La imagen se hace más grande o más pequeña, pero no se deforma.
La escala anisotrópica significa que escalas distinto en X y en Y: La imagen puede quedar estirada o aplastada.


In [ ]:
initial_alignment_uniform = initial_align_he_to_hsi(
    he_oriented_space=he_oriented_space,
    hsi_space=hsi_space,
    use_anisotropic_scale=False,
    show_debug=True,
)